# SB100 Squad 2 — extração com Chandra OCR 2

**Toda página passa pelo Chandra.** Sem Docling: ele nem é instalado, o que
economiza uns 2 minutos por sessão.

Ele lê a página renderizada, então contorna PDF com fonte sem mapeamento.
Foi assim que o Boletim 100 rodou.

Antes de começar: **Ambiente de execução → Alterar tipo → GPU**.

Se a sessão cair, nada do que já subiu se perde. Reabra e rode de novo:
a seleção é por `extracted = false`, então ele traz só o que falta.


## 1. Preparo

Vai pedir o arquivo `.env`, que está em `SB100/squad-2\`. Nenhuma chave
é digitada; o arquivo fica só nesta sessão.


In [ ]:
import os, shutil, torch
from google.colab import files

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "SEM GPU - troque em Ambiente de execucao")

%cd /content
!rm -rf /content/repo
!git clone -q https://github.com/nicolasaws1/parser_rag_framework.git /content/repo
%cd /content/repo
!pip install -q supabase python-dotenv pymupdf

print()
print("Escolha o arquivo .env da pasta SB100/squad-2:")
enviados = files.upload()
nome = next(iter(enviados))
if nome != ".env": shutil.move(f"/content/repo/{nome}", "/content/repo/.env")

linhas = [l for l in open("/content/repo/.env") if "=" in l and not l.startswith("#")]
tem = {l.split("=")[0].strip() for l in linhas}
for k in ("SUPABASE_URL", "SUPABASE_ANON_KEY", "SUPABASE_SERVICE_ROLE_KEY"):
    assert k in tem, f"o .env enviado nao tem {k}"
print(f"ambiente pronto: {len(linhas)} variaveis")


## 2. Ver o que falta

Só relata, não baixa nada.


In [ ]:
!python extractor/baixar_lote.py --quantos 0


## 3. Baixar o lote

Prioriza o que foi pedido pelo botão **Analisar** do site; depois os
menores, que dão retorno rápido e cabem numa sessão sem risco de perder
tudo no meio. Imprime a estimativa de GPU antes de você seguir.


In [ ]:
QUANTOS = 3    #@param {type:"integer"}
MENORES = True #@param {type:"boolean"}

extra = "--menores" if MENORES else ""
!python extractor/baixar_lote.py --quantos {QUANTOS} {extra}


## 4. Extrair

**CHANDRA_MAX_LADO**: lado maior da imagem entregue ao modelo. Era 1800 fixo,
o que encolhia uma A4 a 230 DPI e comia o detalhe fino, justamente rótulo de
eixo e expoente. 2200 usa a VRAM da L4 para ler melhor; o custo cresce com a
área da imagem.

**LOTE_PAGINAS**: páginas por chamada. `1` é o de sempre. **Não testei em
GPU** — rode com 1, veja o tempo impresso no fim, depois tente 4 no mesmo
documento. Se piorar ou quebrar, volte para 1.


In [ ]:
# extrator: chandra
CHANDRA_MAX_LADO = 2200  #@param {type:"integer"}
LOTE_PAGINAS = 1         #@param {type:"integer"}

import os
os.environ["CHANDRA_MAX_LADO"] = str(CHANDRA_MAX_LADO)
os.environ["LOTE_PAGINAS"] = str(LOTE_PAGINAS)
!python extractor/extrator_chandra.py


## 5. Mandar para o Supabase

`ingerir_extracao.py` **atualiza** o documento que já existe e não encosta
em `article_metadata`. Não troque pelo `ingest_supabase.py`: aquele apaga a
linha de `pdfs` e recria, e o cascade levaria junto os metadados vindos da
curadoria, que aqui não há de onde repor.

`gerar_figuras.py` recorta cada figura do PDF pela bbox do banco. É
idempotente, então só faz o que falta.


In [ ]:
!python scripts/ingerir_extracao.py /content/export
!python scripts/gerar_figuras.py --aplicar


## 6. Conferir

Depois disto, abra o site: os documentos do lote aparecem como extraídos,
com as figuras.


In [ ]:
!python scripts/acervo.py 2>/dev/null | head -12


## 7. Guardar uma cópia (opcional)

O resultado já está no Supabase. Isto é só para ter o arquivo bruto da
corrida na sua máquina.


In [ ]:
import shutil
shutil.make_archive('/content/extracao_chandra', 'zip', '/content/export')
from google.colab import files
files.download('/content/extracao_chandra.zip')
